# Day 2 - Exercise 01: Prompt Templates and Chaining Workflows

## Real-World Scenario: Email Processing System

**Context:** You're building an email processing system for a company that:
1. Classifies incoming emails (support, sales, spam, general)
2. Extracts key information
3. Generates appropriate responses
4. Routes to the right department

**Learning Objectives:**
- Master different types of PromptTemplates
- Build multi-step chains with LCEL (LangChain Expression Language)
- Use output parsers for structured data
- Create branching workflows based on conditions

---

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    FewShotPromptTemplate,
    MessagesPlaceholder
)
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from pydantic import BaseModel, Field
from typing import List, Optional

llm = llm = ChatAnthropic(model="global.anthropic.claude-haiku-4-5-20251001-v1:0",api_key=os.environ.get("KEY"),base_url="https://llmgw-wp.tekstac.com", max_tokens=200)
print("Setup complete!")

---
## Part 1: Understanding Prompt Templates

LangChain offers different template types for different use cases:

| Template Type | Use Case | Example |
|--------------|----------|--------|
| `PromptTemplate` | Simple text prompts | Summaries, translations |
| `ChatPromptTemplate` | Multi-turn conversations | Chatbots, agents |
| `FewShotPromptTemplate` | Learning from examples | Classification, formatting |

### 1.1 Basic PromptTemplate

In [ ]:
# Simple PromptTemplate
simple_prompt = PromptTemplate.from_template(
    "Summarize the following email in one sentence:\n\n{email}"
)

# Test it
test_email = """
Hi,
I ordered a laptop last week (Order #12345) but haven't received any shipping confirmation.
Can you please check the status? I need it for a presentation next Monday.
Thanks,
John
"""

chain = simple_prompt | llm | StrOutputParser()
result = chain.invoke({"email": test_email})
print("Summary:", result)

In [ ]:
# ChatPromptTemplate with system and human messages
email_classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an email classifier. Classify emails into one of these categories:
- SUPPORT: Technical issues, product problems, complaints
- SALES: Pricing inquiries, purchase requests, quotes
- SPAM: Promotional, unsolicited, suspicious
- GENERAL: Everything else

Respond with ONLY the category name."""),
    ("human", "Classify this email:\n\n{email}")
])

classifier_chain = email_classifier_prompt | llm | StrOutputParser()

# Test with different emails
emails = [
    "My laptop screen is flickering. Order #12345. Please help!",
    "Can you send me a quote for 50 units of Widget Pro?",
    "Congratulations! You've won $1,000,000! Click here to claim!",
    "Just wanted to say thanks for the great service last week."
]

print("Email Classifications:")
print("-" * 40)
for email in emails:
    category = classifier_chain.invoke({"email": email})
    print(f"{category}: {email[:50]}...")

---
##  Your Turn: Practice Exercises

### Exercise 1.1: Create a Translation Chain

Build a chain that:
1. Detects the language of input text
2. Translates to English if not English
3. Summarizes the content

In [ ]:
# Step 1: Language detection prompt
detect_prompt = ChatPromptTemplate.from_messages([ 
    ("system", "Detect the language of the text. Reply with only the language name (e.g., 'English', 'Spanish', 'French')."), 
    ("human", "{text}") 
]) 

# Step 2: Translation prompt 
translate_prompt = ChatPromptTemplate.from_messages([ 
    ("system", "Translate the following {source_language} text to English. If already English, return as-is."), 
    ("human", "{text}") 
]) 

# Step 3: Summary prompt 
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following English text in a single, concise sentence."),
    ("human", "{translated_text}")
])

# Build the complete sequential chain using LCEL
# We use RunnablePassthrough and dictionaries to pass data forward through the chain steps.
full_chain = (
    # A. Detect language first
    {"text": RunnablePassthrough(), "source_language": detect_prompt | llm | StrOutputParser()}
    # B. Feed the detected language and original text into the translator
    | {"translated_text": translate_prompt | llm | StrOutputParser()}
    # C. Pass the final translation into the summarizer
    | summary_prompt 
    | llm 
    | StrOutputParser()
)

# Test input text
sample_text = "Bonjour, je voudrais réserver une table pour deux personnes ce soir à 20 heures."

# Run the chain
final_summary = full_chain.invoke({"text": sample_text})

print("Final Summary:", final_summary)

### Exercise 1.2: Build a Product Review Analyzer

Create a parallel chain that analyzes product reviews for:
- Overall sentiment (positive/negative/neutral)
- Key pros mentioned
- Key cons mentioned
- Star rating prediction (1-5)

In [ ]:
# Create individual prompts for each analysis dimension
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the sentiment of the review. Respond with exactly one word: Positive, Negative, or Neutral."),
    ("human", "{review}")
])

pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key pros mentioned in this review as bullet points. Be extremely concise."),
    ("human", "{review}")
])

cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key cons mentioned in this review as bullet points. Be extremely concise."),
    ("human", "{review}")
])

rating_prompt = ChatPromptTemplate.from_messages([
    ("system", "Predict the star rating (1 to 5) this user would give based on their review. Respond with only the digit."),
    ("human", "{review}")
])

# Assemble individual components into sub-chains
sentiment_chain = sentiment_prompt | llm | StrOutputParser()
pros_chain = pros_prompt | llm | StrOutputParser()
cons_chain = cons_prompt | llm | StrOutputParser()
rating_chain = rating_prompt | llm | StrOutputParser()

# Create the parallel processing chain
analyzer_chain = RunnableParallel(
    sentiment=sentiment_chain,
    pros=pros_chain,
    cons=cons_chain,
    predicted_rating=rating_chain
)

# Test with the sample review
sample_review = """ 
I bought this laptop 3 months ago. The screen quality is amazing and the battery lasts all day. 
However, the keyboard feels a bit mushy and it runs hot when gaming. For the price, it's decent 
but not exceptional. Would recommend for casual users but not power users. 
"""

analysis_results = analyzer_chain.invoke({"review": sample_review})

# Print out your structured results nicely
print("--- Review Analysis Results ---")
print(f"Predicted Rating: {analysis_results['predicted_rating']}/5 Stars")
print(f"Overall Sentiment: {analysis_results['sentiment']}\n")
print(f"👍 PROS:\n{analysis_results['pros']}\n")
print(f"👎 CONS:\n{analysis_results['cons']}")

### Exercise 1.3: Create a Document Processing Pipeline

Build a pipeline for processing legal documents:
1. Extract key dates and parties involved
2. Identify the document type (contract, agreement, notice, etc.)
3. Summarize main obligations
4. Flag potential risks or concerns

In [ ]:
# 1. Define the structural schema for our legal analysis
class LegalAnalysis(BaseModel):
    document_type: str = Field(description="The identified type of document (e.g., Contract, Service Agreement, Notice).")
    parties: List[str] = Field(description="List of all legal parties involved in this agreement.")
    key_dates: List[str] = Field(description="Important dates mentioned in the text, such as effective dates or deadlines.")
    main_obligations: List[str] = Field(description="A list of the primary duties or payment requirements specified.")
    flagged_risks: List[str] = Field(description="Potential areas of risk or financial exposure (e.g., late fees, penalties, termination clauses).")

# 2. Setup our JSON parser using the Pydantic model
json_parser = JsonOutputParser(pydantic_object=LegalAnalysis)

# 3. Create the prompt template injection point
legal_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert legal assistant. Analyze the incoming text and extract structural data.\n"
        "You must follow the format instructions precisely.\n\n"
        "{format_instructions}"
    )),
    ("human", "Analyze the following document:\n\n{document}")
])

# 4. Bind instructions to prompt variables using LCEL injection
legal_chain = (
    RunnablePassthrough.assign(
        format_instructions=lambda x: json_parser.get_format_instructions()
    )
    | legal_prompt
    | llm
    | json_parser
)

# 5. Execute with the sample text
sample_document = """ 
SERVICE AGREEMENT 
 
This Agreement is entered into as of January 15, 2024, between: 
- TechCorp Inc. ("Service Provider"), located at 123 Tech Street, San Francisco, CA 
- ClientCo LLC ("Client"), located at 456 Business Ave, New York, NY 
 
TERMS: 
1. Service Provider agrees to deliver software development services for a period of 12 months. 
2. Client agrees to pay $50,000 per month, due on the 1st of each month. 
3. Late payments will incur a 5% penalty fee. 
4. Either party may terminate with 30 days written notice. 
5. All intellectual property created shall belong to the Client. 
6. Service Provider shall maintain confidentiality for 5 years after termination. 
 
Signed by both parties on the date first written above. 
"""

analysis = legal_chain.invoke({"document": sample_document})

# 6. Display the parsed dictionary variables cleanly
print("--- Extracted Legal Data (JSON Out) ---")
import json
print(json.dumps(analysis, indent=2))